# 01 — Qwen2.5-1.5B-Instruct reference (Phase 1)

Produces `results/raw/phase1_reference.jsonl`: the correctness oracle and
the reference benchmark every later phase is a delta against.

This is a **re-run**, not a first run. The sweep now labels rows
`attention="gqa"` (the checkpoint has always been GQA — 12 query heads
over 2 KV heads; the old `"mha"` label was describing the runtime in the
architecture column), records the runtime in `extra`, and writes a row
with `status="oom"` instead of silently dropping a context length it
can't fit.

**Budget:** ~10 minutes including the download.

## 0. Session settings (do this in the sidebar before running anything)

* **Accelerator:** GPU T4 x2
* **Internet:** ON (needed to clone the repo and pull the checkpoint)
* **Persistence:** Files only

Everything below pins execution to **GPU 0**. Phases 0-17 are all
single-GPU experiments; the second T4 is Phase 18's subject, and letting
work drift across both would make every earlier number incomparable.

In [ ]:
import os

# Set before torch is imported anywhere, and before any `!python` child
# process inherits the environment.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Two ways in. If you attached the repo as a Kaggle Dataset, point
# UPLOAD at it; otherwise it clones. A private repo needs a PAT in
# Add-ons -> Secrets as GITHUB_TOKEN.
REPO  = "https://github.com/Jash-Vora/LatentServe.git"
DEST  = Path("/kaggle/working/LatentServe")
UPLOAD = Path("/kaggle/input/latentserve")   # optional dataset fallback

if not DEST.exists():
    if UPLOAD.exists():
        !cp -r {UPLOAD} {DEST}
    else:
        token = ""
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass  # public repo, or you're pulling a fork
        url = REPO.replace("https://", f"https://{token}@") if token else REPO
        subprocess.run(["git", "clone", "--depth", "1", url, str(DEST)], check=True)

os.chdir(DEST)
sys.path.insert(0, str(DEST))
os.environ["PYTHONPATH"] = f"{DEST}:" + os.environ.get("PYTHONPATH", "")
print("cwd:", Path.cwd())
!git log --oneline -1

In [ ]:
# Do NOT install torch here. Kaggle ships a build matched to its driver;
# replacing it is the fastest way to lose an afternoon.
!pip install -q pydantic pyyaml 2>&1 | tail -1

import torch, transformers
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0))
print("transformers", transformers.__version__)

major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
if (major, minor) < (4, 40):
    print("\n[!] pyproject requires transformers>=4.40. Run:  !pip install -q -U transformers")
else:
    print("transformers OK (LatentServe's own layer loop is version-independent "
          "by design, but model/qwen.py's HF cache walk needs >=4.40)")

In [ ]:
# Phase 0 gate. Catches a broken driver/CUDA pairing that
# torch.cuda.is_available() alone reports as fine.
!python -m benchmarks.runners.check_env

In [ ]:
# Pull the checkpoint once, up front, so the download (~3 GB) never lands
# inside a timed benchmark. Later sessions re-download unless you save the
# HF cache as a Kaggle Dataset — worth doing once you're iterating daily.
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

cfg = AutoConfig.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
print(f"layers={cfg.num_hidden_layers} q_heads={cfg.num_attention_heads} "
      f"kv_heads={cfg.num_key_value_heads} "
      f"group={cfg.num_attention_heads // cfg.num_key_value_heads} "
      f"head_dim={getattr(cfg, 'head_dim', None) or cfg.hidden_size // cfg.num_attention_heads}")
_ = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
_ = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype="float16")
del _
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("cached")

## 1. Gate 1 — can Qwen generate correctly?

Teacher-forced vs. incremental logits, KV-cache updates, determinism.
The fast subset (1/16/128/1K) runs in a couple of minutes; the gated
long-context tier is what the T4 is for, so run both here.

In [ ]:
!python -m pytest tests/test_phase1_correctness.py -v -x 2>&1 | tail -25

In [ ]:
# Long-context tier (4K/8K/16K). Slow — skip on a quick iteration pass.
!LATENTSERVE_LONG_CONTEXT_TESTS=1 python -m pytest tests/test_phase1_correctness.py -v 2>&1 | tail -20

## 2. The sweep

1 / 16 / 1K / 4K / 8K / 16K, 256 output tokens, 1 warm-up + 3 measured
trials each. 16K is expected to OOM on a 16 GB T4 — HF materialises
logits for every prefill position ([1, 16384, 151936] is 5 GB in fp16
against a 455 MB KV cache). That row gets recorded as a result.

In [ ]:
!python -m benchmarks.runners.phase1_reference \
    --config configs/phase1_reference.yaml \
    --context-lengths 1 16 1024 4096 8192 16384 \
    --output-tokens 256 --repeats 3 --warmup 1

## 3. Read it back

In [ ]:
import pandas as pd

df = pd.read_json("results/raw/phase1_reference.jsonl", lines=True)
status = df["extra"].apply(lambda e: (e or {}).get("status", "ok"))
ok = df[status == "ok"]

print(ok[["context_length", "ttft_ms", "tpot_ms", "throughput_tokens_sec",
          "peak_vram_mb", "kv_cache_mb"]].to_string(index=False))
print("\nnon-ok rows:", list(zip(df.loc[status != "ok", "context_length"], status[status != "ok"])))

# The decode-bandwidth number Phase 2's predictions are built on.
weights_gb = 1.54e9 * 2 / 1e9
for _, r in ok.iterrows():
    kv_gb = r["kv_cache_mb"] / 1024
    bw = (weights_gb + kv_gb) / (r["tpot_ms"] / 1000)
    print(f"ctx={r['context_length']:>6}  tpot={r['tpot_ms']:6.2f}ms  "
          f"{bw:6.1f} GB/s ({bw / 320 * 100:4.1f}% of T4 peak)  "
          f"KV={kv_gb / (kv_gb + weights_gb) * 100:4.1f}% of step bytes")

**What to check before moving on**

* Gate 1 tests green.
* `kv_cache_mb` at 4096 is ~115 MB — that is 28,672 B/token x 4224
  tokens, and it must agree with `ModelShape.kv_bytes_per_token()`.
* Achieved bandwidth well below 320 GB/s at every context (expect
  ~75-80 GB/s). That is prediction P1 in `docs/phase2.md`, and it is
  what decides where the MLA experiments are worth running.

In [ ]:
# results/raw/*.jsonl is gitignored, so it survives only as notebook
# output. Zip it into /kaggle/working before the session ends.
!cd /kaggle/working/LatentServe && zip -qr /kaggle/working/results_phase1.zip results/raw
!ls -la /kaggle/working/*.zip